In [ ]:
import random
import sys
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np

# has to be here for inports to work in vscode
project_root = str(Path.cwd().parent.parent)
if project_root not in sys.path:
    sys.path.append(project_root)
from animal_recognition.src.data.augmentations import get_train_transforms, _MEAN, _STD


def denormalize(tensor):

    img = tensor.numpy().transpose(1, 2, 0)

    mean = np.array(_MEAN)
    std = np.array(_STD)

    img = std * img + mean

    img = np.clip(img, 0, 1)
    return img


IMAGE_FOLDER = Path("../../animal_recognition/data/processed/accepted")
NUM_IMAGES = 10
NUM_AUGS = 10
IMAGE_SIZE = 224

valid_exts = {".jpg", ".jpeg", ".png"}
image_paths = [p for p in IMAGE_FOLDER.rglob("*") if p.is_file() and p.suffix.lower() in valid_exts]

if not image_paths:
    print(f"No images found in {IMAGE_FOLDER}")
else:
    print(f"Found {len(image_paths)} images. Generating preview...")

    selected_paths = random.sample(image_paths, min(NUM_IMAGES, len(image_paths)))

    transform = get_train_transforms(image_size=IMAGE_SIZE)

    fig, axes = plt.subplots(
        len(selected_paths), NUM_AUGS + 1, figsize=(3 * (NUM_AUGS + 1), 3 * len(selected_paths))
    )

    if len(selected_paths) == 1:
        axes = np.expand_dims(axes, axis=0)

    for i, img_path in enumerate(selected_paths):
        original_img = cv2.imread(str(img_path))
        original_img = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)

        ax = axes[i, 0]
        ax.imshow(original_img)
        ax.set_title("Original")
        ax.axis("off")

        for j in range(1, NUM_AUGS + 1):
            augmented_tensor = transform(image=original_img)["image"]

            vis_img = denormalize(augmented_tensor)

            ax = axes[i, j]
            ax.imshow(vis_img)
            ax.set_title(f"Augmented {j}")
            ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
import sys
from pathlib import Path
import random
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_ROOT = Path.cwd().parent.parent
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from animal_recognition.src.data.dataset import AnimalDataset, CLASSES

data_dir = PROJECT_ROOT / "animal_recognition" / "data" / "processed" / "accepted"
dataset = AnimalDataset(root=data_dir, transform=None)

class_indices = {i: [] for i in range(len(CLASSES))}
for idx, (_, label) in enumerate(dataset.samples):
    if label in class_indices:
        class_indices[label].append(idx)

# Randomly sample 10 images per class
for label, indices in class_indices.items():
    if not indices:
        print(f"No images found for class: {CLASSES[label]}")
        continue
        
    sampled_indices = random.sample(indices, min(10, len(indices)))
    
    fig, axes = plt.subplots(1, len(sampled_indices), figsize=(20, 2.5))
    fig.suptitle(f"Class: {CLASSES[label]} (Label: {label})", fontsize=16, y=1.05)
    
    # Handle edge case where there is only 1 image
    if len(sampled_indices) == 1:
        axes = [axes]
        
    for ax, idx in zip(axes, sampled_indices):
        img_path, img_label = dataset.samples[idx]
        
        # Load as PIL Image
        img = Image.open(img_path).convert("RGB")
        
        ax.imshow(img)
        ax.axis("off")
        
    plt.show()
